In [0]:

--List all distinct products sold, along with their total quantity sold.

SELECT
  product,
  SUM(quantity) AS total_quantity_sold
FROM
  samples.bakehouse.sales_transactions
GROUP BY
  product
ORDER BY
  product;

--Find the top 5 franchises by total revenue. 

SELECT
  f.franchiseID,
  f.name,
  SUM(t.totalPrice) AS total_revenue
FROM
  samples.bakehouse.sales_transactions t
JOIN
  samples.bakehouse.sales_franchises f
ON
  t.franchiseID = f.franchiseID
GROUP BY
  f.franchiseID, f.name
ORDER BY
  total_revenue DESC
LIMIT 5;

--Count the number of customers per city.  

SELECT
  city,
  COUNT(customerID) AS customer_count
FROM
  samples.bakehouse.sales_customers
GROUP BY
  city
ORDER BY
  customer_count DESC;


--Find the average transaction amount per product.

SELECT
  product,
  AVG(totalPrice) AS avg_transaction_amount
FROM
  samples.bakehouse.sales_transactions
GROUP BY
  product
ORDER BY
  product;

  --Find month-over-month total sales trend (use date_trunc on the transaction date).
SELECT
  date_trunc('month', dateTime) AS month,
  SUM(totalPrice) AS total_sales
FROM
  samples.bakehouse.sales_transactions
GROUP BY
  date_trunc('month', dateTime)
ORDER BY
  month;

  
--Identify the top 3 best-selling products per franchise using a window function (RANK() or ROW_NUMBER())

WITH ranked_products AS (
  SELECT
    sf.franchiseID,
    sf.name AS franchise_name,
    st.product,
    SUM(st.quantity) AS total_quantity_sold,
    RANK() OVER (
      PARTITION BY sf.franchiseID
      ORDER BY SUM(st.quantity) DESC
    ) AS product_rank
  FROM
    sales_transactions st
  JOIN
    sales_franchises sf
  ON
    st.franchiseID = sf.franchiseID
  GROUP BY
    sf.franchiseID, sf.name, st.product
)
SELECT
  franchiseID,
  franchise_name,
  product,
  total_quantity_sold,
  product_rank
FROM
  ranked_products
WHERE
  product_rank <= 3
ORDER BY
  franchiseID, product_rank




--Find customers who made purchases but never left a review (anti-join between sales_customers and media_customer_reviews).

-- Note: media_customer_reviews does not have customerID column
-- Reviews are tracked at franchise level only, not per customer
-- This query finds customers who purchased from franchises with no reviews
SELECT DISTINCT c.*
FROM samples.bakehouse.sales_customers c
JOIN samples.bakehouse.sales_transactions t
  ON c.customerID = t.customerID
LEFT ANTI JOIN samples.bakehouse.media_customer_reviews r
  ON t.franchiseID = r.franchiseID;


--Calculate the average review rating per product and compare it against sales volume — is there a correlation?

-- Correlation between review ratings and product sales volume
-- Step 1: Extract ratings from reviews per franchise
WITH franchise_ratings AS (
  SELECT
    franchiseID,
    AVG(TRY_CAST(TRIM(SUBSTRING(review, POSITION('**' IN review) + 2, 
                                 POSITION('/5 stars' IN review) - POSITION('**' IN review) - 2)) AS FLOAT)) AS avg_rating,
    COUNT(*) AS review_count
  FROM samples.bakehouse.media_customer_reviews
  WHERE review LIKE '**%/5 stars**%'
  GROUP BY franchiseID
),
-- Step 2: Calculate sales volume per product
product_sales AS (
  SELECT
    product,
    SUM(quantity) AS total_quantity_sold,
    SUM(totalPrice) AS total_revenue,
    COUNT(DISTINCT franchiseID) AS franchises_selling_product
  FROM samples.bakehouse.sales_transactions
  GROUP BY product
),
-- Step 3: Calculate average franchise rating per product (weighted by sales at each franchise)
product_avg_ratings AS (
  SELECT
    t.product,
    AVG(fr.avg_rating) AS avg_franchise_rating,
    SUM(t.quantity) AS total_sales_at_rated_franchises
  FROM samples.bakehouse.sales_transactions t
  JOIN franchise_ratings fr ON t.franchiseID = fr.franchiseID
  GROUP BY t.product
)
-- Step 4: Final correlation analysis
SELECT
  ps.product,
  ps.total_quantity_sold,
  ps.total_revenue,
  ROUND(par.avg_franchise_rating, 2) AS avg_franchise_rating,
  ps.franchises_selling_product,
  ROUND(ps.total_revenue / ps.total_quantity_sold, 2) AS avg_price_per_unit
FROM product_sales ps
LEFT JOIN product_avg_ratings par ON ps.product = par.product
ORDER BY ps.total_quantity_sold DESC;


--Compute a running (cumulative) total of sales per franchise ordered by date, using a window function.


-- Running total of sales per franchise by date
WITH daily_franchise_sales AS (
  SELECT
    t.franchiseID,
    f.name,
    DATE(t.dateTime) AS date,
    SUM(t.totalPrice) AS daily_sales
  FROM samples.bakehouse.sales_transactions t
  JOIN samples.bakehouse.sales_franchises f ON t.franchiseID = f.franchiseID
  GROUP BY t.franchiseID, f.name, DATE(t.dateTime)
)
SELECT
  franchiseID,
  name,
  date,
  daily_sales,
  SUM(daily_sales) OVER (
    PARTITION BY franchiseID
    ORDER BY date
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  ) AS running_total_sales
FROM daily_franchise_sales
ORDER BY franchiseID, date;


--Segment customers into spend tiers (e.g., High/Medium/Low) using NTILE() or CASE WHEN on total lifetime spend.

-- Customer lifetime spend segmentation into High, Medium, Low tiers
WITH customer_lifetime_spend AS (
  SELECT
    c.customerID,
    CONCAT(c.first_name, ' ', c.last_name) AS customer_name,
    SUM(t.totalPrice) AS total_lifetime_spend
  FROM samples.bakehouse.sales_customers c
  JOIN samples.bakehouse.sales_transactions t
    ON c.customerID = t.customerID
  GROUP BY c.customerID, c.first_name, c.last_name
),
tiered_customers AS (
  SELECT
    customerID,
    customer_name,
    total_lifetime_spend,
    NTILE(3) OVER (ORDER BY total_lifetime_spend DESC) AS spend_tier
  FROM customer_lifetime_spend
)
SELECT
  customerID,
  customer_name,
  total_lifetime_spend,
  spend_tier,
  CASE
    WHEN spend_tier = 1 THEN 'High'
    WHEN spend_tier = 2 THEN 'Medium'
    WHEN spend_tier = 3 THEN 'Low'
  END AS spend_tier_label
FROM tiered_customers
ORDER BY total_lifetime_spend DESC;


--Find each franchise's month with the highest sales ("best month") using QUALIFY + ROW_NUMBER().

WITH monthly_sales AS (
  SELECT
    sf.franchiseID,
    sf.name AS franchise_name,
    DATE_TRUNC('month', st.dateTime) AS sales_month,
    SUM(st.totalPrice) AS total_sales
  FROM
    sales_franchises sf
  JOIN
    sales_transactions st
      ON sf.franchiseID = st.franchiseID
  GROUP BY
    sf.franchiseID,
    sf.name,
    DATE_TRUNC('month', st.dateTime)
)
SELECT
  franchiseID,
  franchise_name,
  sales_month,
  total_sales
FROM monthly_sales
QUALIFY
  ROW_NUMBER() OVER (
    PARTITION BY franchiseID
    ORDER BY total_sales DESC
  ) = 1;



--NYC Taxi questions

--Find the average fare_amount and trip_distance overall.

SELECT
  AVG(fare_amount) AS avg_fare_amount,
  AVG(trip_distance) AS avg_trip_distance
FROM samples.nyctaxi.trips


--Count total trips per day.

SELECT
  DATE(tpep_pickup_datetime) AS trip_date,
  COUNT(*) AS total_trips
FROM samples.nyctaxi.trips
GROUP BY DATE(tpep_pickup_datetime)
ORDER BY trip_date


--Find the top 10 pickup zip codes by number of trips.

SELECT
  pickup_zip,
  COUNT(*) AS trip_count
FROM
  samples.nyctaxi.trips
GROUP BY
  pickup_zip
ORDER BY
  trip_count DESC
LIMIT 10


--Find the longest and shortest trips by distance.

SELECT * FROM (
  SELECT
    'Longest Trip' AS trip_type,
    tpep_pickup_datetime,
    tpep_dropoff_datetime,
    trip_distance,
    fare_amount,
    pickup_zip,
    dropoff_zip
  FROM samples.nyctaxi.trips
  ORDER BY trip_distance DESC
  LIMIT 1
)

UNION ALL

SELECT * FROM (
  SELECT
    'Shortest Trip' AS trip_type,
    tpep_pickup_datetime,
    tpep_dropoff_datetime,
    trip_distance,
    fare_amount,
    pickup_zip,
    dropoff_zip
  FROM samples.nyctaxi.trips
  ORDER BY trip_distance ASC
  LIMIT 1
);


--Calculate average fare by hour of day (extract hour from tpep_pickup_datetime) to find peak pricing times.

SELECT
  HOUR(tpep_pickup_datetime) AS pickup_hour,
  ROUND(AVG(fare_amount), 2) AS avg_fare,
  COUNT(*) AS trip_count
FROM samples.nyctaxi.trips
GROUP BY HOUR(tpep_pickup_datetime)
ORDER BY pickup_hour;


--Compute trip duration (dropoff - pickup) and find its correlation with fare_amount.

WITH trip_durations AS (
  SELECT
    fare_amount,
    (UNIX_TIMESTAMP(tpep_dropoff_datetime) - UNIX_TIMESTAMP(tpep_pickup_datetime)) / 60 AS duration_minutes,
    trip_distance
  FROM samples.nyctaxi.trips
  WHERE tpep_dropoff_datetime > tpep_pickup_datetime
    AND fare_amount > 0
)
SELECT
  ROUND(CORR(duration_minutes, fare_amount), 4) AS correlation_duration_fare,
  ROUND(AVG(duration_minutes), 2) AS avg_duration_minutes,
  ROUND(AVG(fare_amount), 2) AS avg_fare,
  COUNT(*) AS total_trips
FROM trip_durations;


--Find the busiest pickup zip → dropoff zip pairs (top routes by trip count).

SELECT
  pickup_zip,
  dropoff_zip,
  COUNT(*) AS trip_count,
  ROUND(AVG(trip_distance), 2) AS avg_distance,
  ROUND(AVG(fare_amount), 2) AS avg_fare
FROM samples.nyctaxi.trips
WHERE pickup_zip IS NOT NULL
  AND dropoff_zip IS NOT NULL
GROUP BY pickup_zip, dropoff_zip
ORDER BY trip_count DESC
LIMIT 20;

--Flag anomalies: trips with trip_distance = 0 but fare_amount > 0, or very high fare-per-mile.


-- Anomaly Type 1: Zero distance but positive fare
SELECT
  'Zero Distance, Positive Fare' AS anomaly_type,
  tpep_pickup_datetime,
  tpep_dropoff_datetime,
  trip_distance,
  fare_amount,
  pickup_zip,
  dropoff_zip,
  NULL AS fare_per_mile
FROM samples.nyctaxi.trips
WHERE trip_distance = 0
  AND fare_amount > 0

UNION ALL

-- Anomaly Type 2: Very high fare per mile (>$50/mile)
SELECT
  'High Fare Per Mile' AS anomaly_type,
  tpep_pickup_datetime,
  tpep_dropoff_datetime,
  trip_distance,
  fare_amount,
  pickup_zip,
  dropoff_zip,
  ROUND(fare_amount / trip_distance, 2) AS fare_per_mile
FROM samples.nyctaxi.trips
WHERE trip_distance > 0
  AND (fare_amount / trip_distance) > 50

ORDER BY anomaly_type, fare_amount DESC;


--Compare weekday vs weekend average trip volume and fares.

WITH trip_classification AS (
  SELECT
    CASE 
      WHEN DAYOFWEEK(tpep_pickup_datetime) IN (1, 7) THEN 'Weekend'
      ELSE 'Weekday'
    END AS day_type,
    DATE(tpep_pickup_datetime) AS trip_date,
    fare_amount,
    trip_distance
  FROM samples.nyctaxi.trips
  WHERE tpep_pickup_datetime IS NOT NULL
    AND fare_amount > 0
),
daily_stats AS (
  SELECT
    day_type,
    trip_date,
    COUNT(*) AS daily_trip_count,
    AVG(fare_amount) AS daily_avg_fare,
    AVG(trip_distance) AS daily_avg_distance
  FROM trip_classification
  GROUP BY day_type, trip_date
)
SELECT
  day_type,
  ROUND(AVG(daily_trip_count), 0) AS avg_daily_trips,
  ROUND(AVG(daily_avg_fare), 2) AS avg_fare,
  ROUND(AVG(daily_avg_distance), 2) AS avg_distance,
  COUNT(DISTINCT trip_date) AS total_days
FROM daily_stats
GROUP BY day_type
ORDER BY day_type;


--Use a window function to rank zip codes by daily trip count and find each day's top pickup zone.


WITH daily_zip_counts AS (
  SELECT
    DATE(tpep_pickup_datetime) AS trip_date,
    pickup_zip,
    COUNT(*) AS trip_count
  FROM samples.nyctaxi.trips
  WHERE pickup_zip IS NOT NULL
  GROUP BY DATE(tpep_pickup_datetime), pickup_zip
),
ranked_zips AS (
  SELECT
    trip_date,
    pickup_zip,
    trip_count,
    ROW_NUMBER() OVER (PARTITION BY trip_date ORDER BY trip_count DESC) AS rank
  FROM daily_zip_counts
)
SELECT
  trip_date,
  pickup_zip,
  trip_count
FROM ranked_zips
WHERE rank = 1
ORDER BY trip_date;


--Build an hourly heatmap query: trips grouped by hour and day_of_week (useful later for a chart).

SELECT
  DAYOFWEEK(tpep_pickup_datetime) AS day_of_week_num,
  DAYNAME(tpep_pickup_datetime) AS day_of_week,
  HOUR(tpep_pickup_datetime) AS hour_of_day,
  COUNT(*) AS trip_count,
  ROUND(AVG(fare_amount), 2) AS avg_fare,
  ROUND(AVG(trip_distance), 2) AS avg_distance
FROM samples.nyctaxi.trips
WHERE tpep_pickup_datetime IS NOT NULL
GROUP BY DAYOFWEEK(tpep_pickup_datetime), DAYNAME(tpep_pickup_datetime), HOUR(tpep_pickup_datetime)
ORDER BY day_of_week_num, hour_of_day;

--Calculate a rolling 7-day average of daily fare revenue using AVG() OVER (ORDER BY ... ROWS BETWEEN 6 PRECEDING AND CURRENT ROW).

WITH daily_revenue AS (
  SELECT
    DATE(tpep_pickup_datetime) AS trip_date,
    SUM(fare_amount) AS daily_fare_revenue,
    COUNT(*) AS trip_count
  FROM samples.nyctaxi.trips
  WHERE tpep_pickup_datetime IS NOT NULL
    AND fare_amount > 0
  GROUP BY DATE(tpep_pickup_datetime)
)
SELECT
  trip_date,
  daily_fare_revenue,
  trip_count,
  ROUND(AVG(daily_fare_revenue) OVER (
    ORDER BY trip_date
    ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
  ), 2) AS rolling_7day_avg_revenue,
  COUNT(*) OVER (
    ORDER BY trip_date
    ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
  ) AS days_in_window
FROM daily_revenue
ORDER BY trip_date;


--Identify outlier trips using standard deviation (e.g., fares more than 3 std devs from the mean).

WITH fare_stats AS (
  SELECT
    AVG(fare_amount) AS mean_fare,
    STDDEV(fare_amount) AS stddev_fare
  FROM samples.nyctaxi.trips
  WHERE fare_amount > 0
),
trips_with_stats AS (
  SELECT
    t.tpep_pickup_datetime,
    t.tpep_dropoff_datetime,
    t.trip_distance,
    t.fare_amount,
    t.pickup_zip,
    t.dropoff_zip,
    s.mean_fare,
    s.stddev_fare,
    ROUND(ABS(t.fare_amount - s.mean_fare) / s.stddev_fare, 2) AS std_devs_from_mean
  FROM samples.nyctaxi.trips t
  CROSS JOIN fare_stats s
  WHERE t.fare_amount > 0
)
SELECT
  tpep_pickup_datetime,
  tpep_dropoff_datetime,
  trip_distance,
  fare_amount,
  pickup_zip,
  dropoff_zip,
  ROUND(mean_fare, 2) AS mean_fare,
  ROUND(stddev_fare, 2) AS stddev_fare,
  std_devs_from_mean,
  CASE
    WHEN fare_amount > mean_fare THEN 'High Outlier'
    ELSE 'Low Outlier'
  END AS outlier_type
FROM trips_with_stats
WHERE std_devs_from_mean > 3
ORDER BY std_devs_from_mean DESC;
